# 09 · The sealed test: open it once, report it honestly: BankSight

**What this is.** Every choice so far (features, tuning, champion, calibration, threshold) was made on train and validation. The test period (complaints received after 2025-11-01)
has been locked away. This notebook opens it **once** to answer one question: *does the champion still work on data it has never influenced?*

**What this does:** loads the libraries.

In [1]:
from datetime import datetime
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

**What this does:** sets file locations and the settings. The business numbers come from the saved champion file, so they cannot drift from what 08f saved.

In [2]:
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
WAREHOUSE = ROOT / "data" / "warehouse" / "banksight.duckdb"
MODEL_DIR = ROOT / "models"
REPORT_DIR = ROOT / "reports" / "test"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TEST_PRED_FILE = ROOT / "data" / "colab" / "outputs" / "test_predictions_roberta.csv"
LOCK_FILE = MODEL_DIR / "09_test_opened.txt"
N_BOOT = 1000
SEED = 42
OPENING_CHARS = 150
MIN_REPEATS = 5
TOP_SHARES = [0.05, 0.10, 0.20, 0.30]

assert TEST_PRED_FILE.exists(), "Run 08e on Colab and copy test_predictions_roberta.csv into data/colab/outputs/."
for needed in ["04_champion_bundle.joblib", "07_topic_model.joblib", "08_champion_roberta.joblib"]:
    assert (MODEL_DIR / needed).exists(), f"Missing models/{needed}. Run the earlier notebooks first."

**What this does:** the lock. If the test was already opened, it stops here. Delete the lock file only in one case: the notebook crashed on a **setup error before printing any test result** (for example a missing file). Never delete it to run the test again after seeing results.

In [3]:
if LOCK_FILE.exists():
    raise SystemExit("The test set was already opened: " + LOCK_FILE.read_text())

LOCK_FILE.write_text("opened on " + datetime.now().strftime("%Y-%m-%d %H:%M") + ". Do not tune anything after this.")
print("lock written. Test opened.")

lock written. Test opened.


## 1. Load everything

**What this does:** reads every complaint with a final outcome (train, validation and test). The text of all three is needed to find repeated wording; the test **labels** are used only from the next sections on.

In [4]:
con = duckdb.connect(str(WAREHOUSE), read_only=True)
df = con.sql("""
    SELECT l.complaint_id, l.date_received, l.product_group, l.got_relief, l.split, r.issue, r.narrative
    FROM features.complaint_labels l
    JOIN raw.cfpb_complaints r USING (complaint_id)
    WHERE l.target_eligible
      AND r.narrative IS NOT NULL AND length(trim(r.narrative)) > 0
""").df()
con.close()

df["issue"] = df["issue"].fillna("Unknown").astype(str)
df["complaint_id"] = df["complaint_id"].astype(str)
df["date_received"] = pd.to_datetime(df["date_received"])
df = df.sort_values("date_received", kind="stable").reset_index(drop=True)
print(df["split"].value_counts().to_string())

split
train         103777
test           26452
validation     18537


**What this does:** cleans the text exactly as in notebooks 04, 07 and 08c (amounts, dates and masks replaced by tags).

In [5]:
clean_text = df["narrative"].str.replace(r"\{\$[\d,\.]+\}|\$\s?[\d,]+(?:\.\d+)?", " amtmask ", regex=True)
clean_text = clean_text.str.replace(r"\bXX/XX/(?:XXXX|\d{2,4})\b|\bXX/\d{2}/\d{2,4}\b", " datemask ", regex=True)
clean_text = clean_text.str.replace(r"\bX{2,}\b", " xmask ", regex=True)
df["text"] = clean_text.str.replace(r"\s+", " ", regex=True).str.strip()

is_train = (df["split"] == "train").to_numpy()
is_test = (df["split"] == "test").to_numpy()
test = df[is_test].copy().reset_index(drop=True)
print(f"test complaints: {len(test):,} | {test['date_received'].min().date()} to {test['date_received'].max().date()}")

test complaints: 26,452 | 2025-11-01 to 2026-07-29


## 2. Scores from each model

**What this does:** B1 (relief rate of the product, learned from train only) and B4 (the old champion from notebook 04). Both were fitted before the test was opened.

In [6]:
bundle = joblib.load(MODEL_DIR / "04_champion_bundle.joblib")

X_test = sp.hstack([bundle["tfidf"].transform(test["text"]), bundle["ohe"].transform(test[["product_group", "issue"]])]).tocsr()
test["p_b4"] = bundle["b4_model"].predict_proba(X_test)[:, 1]

product_rate_train = df[is_train].groupby("product_group")["got_relief"].mean()
test["p_b1"] = test["product_group"].map(product_rate_train).fillna(df[is_train]["got_relief"].mean())
print("B1 and B4 scored")

B1 and B4 scored


**What this does:** attaches RoBERTa's test probabilities from Colab, matched on complaint id. It stops if any test complaint has no prediction.

In [7]:
roberta = pd.read_csv(TEST_PRED_FILE)
roberta["complaint_id"] = roberta["complaint_id"].astype(str)
roberta = roberta.rename(columns={"p_relief": "p_roberta"})

test = test.merge(roberta, on="complaint_id", how="left")
assert test["p_roberta"].notna().all(), "Some test complaints have no RoBERTa prediction. Re-run 08d and 08e so they cover the same rows."
y = test["got_relief"].astype(int).to_numpy()
print(f"RoBERTa scored {len(test):,} complaints | test relief rate {y.mean():.3f}")

RoBERTa scored 26,452 complaints | test relief rate 0.183


## 3. The headline: does RoBERTa still beat B4?

**What this does:** PR-AUC, ROC-AUC and within-product ROC-AUC (average over products with at least 100 test complaints, weighted by size), for the three models.

In [8]:
test_models = {"B1 product rate": "p_b1", "B4 (old champion)": "p_b4", "RoBERTa (champion)": "p_roberta"}

rows = []
for model_name, column in test_models.items():
    within_values = []
    within_weights = []
    for product, part in test.groupby("product_group"):
        y_part = part["got_relief"].astype(int).to_numpy()
        if len(part) >= 100 and 0 < y_part.sum() < len(part):
            within_values.append(roc_auc_score(y_part, part[column]))
            within_weights.append(len(part))
    rows.append({"model": model_name, "pr_auc": average_precision_score(y, test[column]), "roc_auc": roc_auc_score(y, test[column]),
                 "within_product_roc_auc": np.average(within_values, weights=within_weights)})

headline = pd.DataFrame(rows).set_index("model")
print(f"random-model PR-AUC would be {y.mean():.3f}")
display(headline.round(4))
headline.to_csv(REPORT_DIR / "09_headline.csv")

random-model PR-AUC would be 0.183


,pr_auc,roc_auc,within_product_roc_auc
model,,,
B1 product rate,0.2652,0.6474,0.5000
B4 (old champion),0.3779,0.7398,0.6882
RoBERTa (champion),0.4027,0.7586,0.7181


**What this does:** the one pre-declared test. It resamples the test complaints 1,000 times and each time recomputes RoBERTa's PR-AUC minus B4's.
If the 95% interval is entirely above 0, the claim "RoBERTa beats B4" is confirmed on unseen data.

In [9]:
rng = np.random.default_rng(SEED)
n = len(test)
p_b4_all = test["p_b4"].to_numpy()
p_rob_all = test["p_roberta"].to_numpy()

gains = []
for repeat in range(N_BOOT):
    pick = rng.integers(0, n, n)
    y_boot = y[pick]
    if y_boot.min() == y_boot.max():
        continue
    gains.append(average_precision_score(y_boot, p_rob_all[pick]) - average_precision_score(y_boot, p_b4_all[pick]))

gains = np.array(gains)
low, high = np.percentile(gains, [2.5, 97.5])
mean_gain = average_precision_score(y, p_rob_all) - average_precision_score(y, p_b4_all)
print(f"RoBERTa minus B4, PR-AUC on test: {mean_gain:+.4f} (95% interval {low:+.4f} to {high:+.4f})")

if low > 0:
    verdict = "CONFIRMED: RoBERTa beats B4 on the sealed test."
elif high < 0:
    verdict = "REVERSED: B4 is better than RoBERTa on the sealed test. Report this honestly; do not re-tune."
else:
    verdict = "NOT CONFIRMED: the interval includes 0. Report this honestly; do not re-tune."
print(verdict)

pd.DataFrame([{"gain": mean_gain, "ci_low": low, "ci_high": high, "verdict": verdict}]).to_csv(REPORT_DIR / "09_bootstrap.csv", index=False)

RoBERTa minus B4, PR-AUC on test: +0.0249 (95% interval +0.0156 to +0.0334)
CONFIRMED: RoBERTa beats B4 on the sealed test.


## 4. Where templates cannot help

**What this does:** flags repeated wording: a narrative whose first 150 characters appear 5 or more times across all complaints. (Only text is used here, no outcomes.)

In [10]:
df["opening"] = df["text"].str.slice(0, OPENING_CHARS)
opening_counts = df["opening"].map(df["opening"].value_counts())
df["repeated_wording"] = opening_counts >= MIN_REPEATS
test["repeated_wording"] = df.loc[is_test, "repeated_wording"].to_numpy()
print(f"repeated wording: {test['repeated_wording'].mean():.1%} of test complaints")

repeated wording: 6.6% of test complaints


**What this does:** finds each complaint's main topic with the saved topic model from notebook 07. The "ordinary-stories" topic is the biggest topic **in the training data**, as in 08c.

In [11]:
topic_bundle = joblib.load(MODEL_DIR / "07_topic_model.joblib")
topic_weights = topic_bundle["nmf"].transform(topic_bundle["vectorizer"].transform(df["text"]))
df["topic"] = np.where(topic_weights.sum(axis=1) > 0, topic_weights.argmax(axis=1), -1)
test["topic"] = df.loc[is_test, "topic"].to_numpy()

organic_topic = df[is_train]["topic"].value_counts().idxmax()
print("ordinary-stories topic:", topic_bundle["topic_labels"][organic_topic])

ordinary-stories topic: T2: general bank/card/money stories


**What this does:** PR-AUC and ROC-AUC of the three models on the whole test set and on the two slices.

In [12]:
subsets = {
    "all test": np.ones(len(test), dtype=bool),
    "non-repeated wording": (~test["repeated_wording"]).to_numpy(),
    "ordinary-stories topic": (test["topic"] == organic_topic).to_numpy(),
}

rows = []
for subset_name, mask in subsets.items():
    y_sub = y[mask]
    if mask.sum() < 300 or y_sub.min() == y_sub.max():
        continue
    for model_name, column in test_models.items():
        rows.append({"subset": subset_name, "complaints": int(mask.sum()), "relief_rate": y_sub.mean(), "model": model_name,
                     "pr_auc": average_precision_score(y_sub, test.loc[mask, column]),
                     "roc_auc": roc_auc_score(y_sub, test.loc[mask, column])})

by_subset = pd.DataFrame(rows)
display(by_subset.pivot(index="model", columns="subset", values="pr_auc").round(4))
display(by_subset.pivot(index="model", columns="subset", values="roc_auc").round(4))
by_subset.to_csv(REPORT_DIR / "09_by_subset.csv", index=False)

subset,all test,non-repeated wording,ordinary-stories topic
model,,,
B1 product rate,0.2652,0.2550,0.2402
B4 (old champion),0.3779,0.3903,0.3957
RoBERTa (champion),0.4027,0.4108,0.4112


subset,all test,non-repeated wording,ordinary-stories topic
model,,,
B1 product rate,0.6474,0.6454,0.6431
B4 (old champion),0.7398,0.7464,0.7585
RoBERTa (champion),0.7586,0.7667,0.7765


## 5. Are the probabilities honest on the test period?
The calibrators were fitted on validation and are **not** re-fitted. If the relief rate moved between validation and test, we will see it here.

**What this does:** applies each model's saved calibrator (B4 from notebook 04, RoBERTa from 08f) to the test scores.

In [13]:
champion = joblib.load(MODEL_DIR / "08_champion_roberta.joblib")

calibrators = {"B4 (old champion)": (bundle["calibration"], p_b4_all), "RoBERTa (champion)": (champion["calibration"], p_rob_all)}
calibrated = {}

for model_name, (cal, raw) in calibrators.items():
    raw_clip = np.clip(raw, 1e-4, 1 - 1e-4)
    if cal["method"] == "platt":
        calibrated[model_name] = cal["platt"].predict_proba(np.log(raw_clip / (1 - raw_clip)).reshape(-1, 1))[:, 1]
    elif cal["method"] == "isotonic":
        calibrated[model_name] = cal["isotonic"].predict(raw)
    else:
        calibrated[model_name] = raw
    print(model_name, "calibration used:", cal["method"])

B4 (old champion) calibration used: isotonic
RoBERTa (champion) calibration used: isotonic


**What this does:** compares raw and calibrated probabilities with Brier score and ECE (the average gap between predicted and observed relief rate over 10 equal-sized bins; lower is better), and shows average predicted against actual rate.

In [14]:
rows = []
for model_name, (cal, raw) in calibrators.items():
    for version, probs in [("raw", raw), ("calibrated", calibrated[model_name])]:
        q = np.clip(probs, 1e-6, 1 - 1e-6)
        frac_obs, mean_pred_bin = calibration_curve(y, q, n_bins=10, strategy="quantile")
        rows.append({"model": model_name, "version": version, "brier": brier_score_loss(y, q),
                     "ece": float(np.mean(np.abs(frac_obs - mean_pred_bin))), "mean_predicted": q.mean(), "actual_rate": y.mean()})

cal_report = pd.DataFrame(rows)
display(cal_report.round(4))
cal_report.to_csv(REPORT_DIR / "09_calibration.csv", index=False)

,model,version,brier,ece,mean_predicted,actual_rate
0,B4 (old champion),raw,0.1348,0.0229,0.2018,0.1834
1,B4 (old champion),calibrated,0.1334,0.0125,0.1867,0.1834
2,RoBERTa (champion),raw,0.1366,0.0577,0.1842,0.1834
3,RoBERTa (champion),calibrated,0.1303,0.0162,0.1813,0.1834


## 6. The decision rule and lift

**What this does:** applies the saved 25% threshold (break-even for `COST_FP` 1 and `BENEFIT_TP` 3) to the calibrated probabilities. We report how many complaints get flagged, the share of flagged that got relief (precision), the share of all relief cases caught (recall) and net value per complaint, next to the two trivial strategies.

In [15]:
threshold = champion["threshold"]
cost_fp = champion["cost_fp"]
benefit_tp = champion["benefit_tp"]

rows = []
for model_name in calibrators:
    flag = calibrated[model_name] >= threshold
    tp = int((flag & (y == 1)).sum())
    fp = int((flag & (y == 0)).sum())
    rows.append({"strategy": model_name, "flagged_share": flag.mean(), "precision": tp / max(tp + fp, 1), "recall": tp / y.sum(),
                 "net_value_per_complaint": (benefit_tp * tp - cost_fp * fp) / len(y)})

rows.append({"strategy": "flag everybody", "flagged_share": 1.0, "precision": y.mean(), "recall": 1.0,
             "net_value_per_complaint": (benefit_tp * y.sum() - cost_fp * (len(y) - y.sum())) / len(y)})
rows.append({"strategy": "flag nobody", "flagged_share": 0.0, "precision": np.nan, "recall": 0.0, "net_value_per_complaint": 0.0})

decision = pd.DataFrame(rows).set_index("strategy")
print(f"threshold {threshold:.2f} on the calibrated probability")
display(decision.round(4))
decision.to_csv(REPORT_DIR / "09_decision_rule.csv")

threshold 0.25 on the calibrated probability


,flagged_share,precision,recall,net_value_per_complaint
strategy,,,,
B4 (old champion),0.2706,0.3568,0.5266,0.1156
RoBERTa (champion),0.3207,0.3590,0.6278,0.1398
flag everybody,1.0000,0.1834,1.0000,-0.2666
flag nobody,0.0000,NaN,0.0000,0.0000


**What this does:** lift. If reviewers can only look at the top k% of complaints ranked by the model, what share of all relief cases do they catch? A random list would catch k%.

In [16]:
rows = []
for model_name, column in [("B4 (old champion)", "p_b4"), ("RoBERTa (champion)", "p_roberta")]:
    order = np.argsort(-test[column].to_numpy())
    for share in TOP_SHARES:
        top = order[:int(round(share * len(test)))]
        caught = y[top].sum() / y.sum()
        rows.append({"model": model_name, "reviewed_top_share": share, "relief_cases_caught": caught, "lift_vs_random": caught / share})

lift = pd.DataFrame(rows)
display(lift.pivot(index="reviewed_top_share", columns="model", values="relief_cases_caught").round(3))
lift.to_csv(REPORT_DIR / "09_lift.csv", index=False)

model,B4 (old champion),RoBERTa (champion)
reviewed_top_share,,
0.05,0.132,0.137
0.10,0.242,0.239
0.20,0.424,0.438
0.30,0.565,0.601


## 7. Safety check: the model must never auto-deny
In notebook 07b we saw topics with almost no relief (for example Cash App and Zelle). The model is a **triage** tool: it decides which complaints get a closer human look first, and it never decides that a customer gets nothing. This cell shows the lowest-scored complaints and how many of them really did get relief, so you can see that "low score" does not mean "no relief".

In [17]:
lowest = np.argsort(test["p_roberta"].to_numpy())[:int(0.10 * len(test))]
print(f"lowest-scored 10% of test complaints: {len(lowest):,}")
print(f"  of these, share that really got relief: {y[lowest].mean():.1%} (overall test rate {y.mean():.1%})")
print(f"  number of relief cases in this group: {int(y[lowest].sum()):,}")

lowest-scored 10% of test complaints: 2,645
  of these, share that really got relief: 2.3% (overall test rate 18.3%)
  number of relief cases in this group: 61


## What to do next
1. **Paste the headline table, the bootstrap line, the two slice tables, the calibration table, the decision table and the lift table.** I will read them with you.
2. Whatever the result, **the champion stays as it is**: if the gain shrinks, we say so in the model card. A smaller honest number is worth more to a recruiter than a tuned one.
3. Next is step 5: the FastAPI service that loads the RoBERTa champion and falls back to B4.

## How to explain this in an interview
> "I treated the test set like a one-shot exam. I wrote the pass rule and a lock file before opening it, and I re-used the calibrators and the threshold exactly as fitted on validation. I report calibration drift and the slices where templates can't help, and I never re-tune after seeing test results."